# 03 · EDA full-cohort y control de calidad físico

Audita **todos los NIfTI de entrenamiento** sin remuestrear ni modificar las fuentes. El
resultado es un manifiesto por examen, familias de adquisición reproducibles y scores de
revisión técnica. La geometría se trata como QC/dominio, no como señal clínica.

> Privacidad: ejecutar sólo localmente. El notebook se versiona sin outputs. Los artefactos
> de `outputs/private_eda/` contienen derivados privados y no deben compartirse.

## 1. Configuración y contrato

`MAX_SCANS=None` procesa la cohorte completa. La orientación RAS sólo homogeneiza ejes;
todavía no constituye registro anatómico. `gradient_energy` se calcula por milímetro y
dentro de voxeles positivos, corrigiendo la versión inicial por-voxel.

In [ ]:
from __future__ import annotations

import json
import tempfile
import zipfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import ndimage
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

NIFTI_ARCHIVE = PROJECT_ROOT / 'data' / 'raw' / 'niftis.zip'
LABELS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train_labels.csv'
PRIVATE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'private_eda'

MAX_SCANS: int | None = None
RANDOM_SEED = 20260821
MIN_PROTOCOL_FAMILY_SIZE = 5
SAVE_PRIVATE_OUTPUTS = True

for required_path in [NIFTI_ARCHIVE, LABELS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

sns.set_theme(style='whitegrid')
display(Markdown(
    f'**Modo:** cohorte completa (`MAX_SCANS={MAX_SCANS}`) · '
    f'**salida privada:** `{PRIVATE_OUTPUT_DIR}`'
))

In [ ]:
def uid_from_member(member_name: str) -> str:
    name = Path(member_name).name
    return name[:-7] if name.lower().endswith('.nii.gz') else Path(name).stem


def list_nifti_members(archive_path: Path) -> list[str]:
    with zipfile.ZipFile(archive_path) as archive:
        return sorted(
            name for name in archive.namelist()
            if name.lower().endswith(('.nii', '.nii.gz')) and not name.endswith('/')
        )


def load_canonical_volume(
    member_name: str,
) -> tuple[np.ndarray, nib.Nifti1Image, dict[str, object]]:
    with tempfile.TemporaryDirectory(prefix='dat_qc_') as temporary_directory:
        with zipfile.ZipFile(NIFTI_ARCHIVE) as archive:
            extracted_path = Path(archive.extract(member_name, temporary_directory))
        image = nib.load(extracted_path)
        original_metadata = {
            'original_orientation': ''.join(nib.aff2axcodes(image.affine)),
            'original_affine_determinant': float(np.linalg.det(image.affine[:3, :3])),
            'original_qform_code': int(image.header['qform_code']),
            'original_sform_code': int(image.header['sform_code']),
            'original_dtype': str(image.get_data_dtype()),
        }
        canonical = nib.as_closest_canonical(image)
        volume = np.asarray(canonical.dataobj, dtype=np.float32).squeeze()
        if volume.ndim != 3:
            raise ValueError(f'{member_name}: se esperaba 3D y se obtuvo {volume.shape}.')
        detached = nib.Nifti1Image(volume, canonical.affine, canonical.header.copy())
    return volume, detached, original_metadata


members = list_nifti_members(NIFTI_ARCHIVE)
if MAX_SCANS is not None and MAX_SCANS < len(members):
    rng = np.random.default_rng(RANDOM_SEED)
    indices = np.sort(rng.choice(len(members), size=MAX_SCANS, replace=False))
    members = [members[index] for index in indices]

member_by_uid = {uid_from_member(member): member for member in members}
display(Markdown(f'**NIfTI seleccionados:** `{len(members):,}`'))

## 2. Manifiesto físico, intensidad y QC

La asimetría se calcula sobre captación por encima de la mediana y no se asigna lateralidad
anatómica antes del registro. `gradient_energy` es la magnitud media del gradiente físico
(intensidad normalizada por milímetro) dentro del soporte positivo.

In [ ]:
def safe_entropy(values: np.ndarray, bins: int = 64) -> float:
    if values.size < 2 or float(values.max()) <= float(values.min()):
        return 0.0
    counts, _ = np.histogram(values, bins=bins)
    probabilities = counts[counts > 0] / counts.sum()
    return float(-(probabilities * np.log2(probabilities)).sum())


def adjacent_slice_correlation(volume01: np.ndarray) -> float:
    correlations: list[float] = []
    for index in range(volume01.shape[2] - 1):
        first = volume01[:, :, index].ravel()
        second = volume01[:, :, index + 1].ravel()
        if first.std() > 1e-6 and second.std() > 1e-6:
            correlations.append(float(np.corrcoef(first, second)[0, 1]))
    return float(np.nanmedian(correlations)) if correlations else float('nan')


def scan_record(member_name: str) -> dict[str, object]:
    volume, image, original_metadata = load_canonical_volume(member_name)
    finite_mask = np.isfinite(volume)
    finite = volume[finite_mask]
    if finite.size == 0:
        raise ValueError(f'{member_name}: no contiene voxels finitos.')

    positive = finite[finite > 0]
    reference = positive if positive.size else finite
    p01, p05, p50, p90, p95, p995 = np.percentile(
        reference, [1, 5, 50, 90, 95, 99.5]
    )
    scale = max(float(p995 - p01), np.finfo(np.float32).eps)
    normalized = np.clip(
        (np.nan_to_num(volume, nan=float(p01)) - p01) / scale, 0, 1
    )
    spacing = tuple(float(value) for value in image.header.get_zooms()[:3])
    shape = tuple(int(value) for value in volume.shape)
    fov = tuple(shape[index] * spacing[index] for index in range(3))

    foreground = finite_mask & (volume > p50)
    high_uptake = finite_mask & (volume > p90)
    labels_cc, component_count = ndimage.label(high_uptake)
    component_sizes = (
        np.bincount(labels_cc.ravel())[1:] if component_count else np.array([])
    )
    largest_fraction = (
        float(component_sizes.max() / max(high_uptake.sum(), 1))
        if component_sizes.size else 0.0
    )

    weights = np.where(foreground, normalized, 0.0)
    centroid_voxel = np.asarray(ndimage.center_of_mass(weights), dtype=float)
    if not np.all(np.isfinite(centroid_voxel)):
        centroid_voxel = (np.asarray(volume.shape, dtype=float) - 1) / 2
    centroid_mm = nib.affines.apply_affine(image.affine, centroid_voxel)

    midpoint = volume.shape[0] // 2
    lower_x_signal = float(weights[:midpoint].sum())
    upper_x_signal = float(weights[-midpoint:].sum())
    denominator = max((lower_x_signal + upper_x_signal) / 2, 1e-8)
    asymmetry_abs = abs(upper_x_signal - lower_x_signal) / denominator
    asymmetry_signed = (upper_x_signal - lower_x_signal) / denominator

    gradients = np.gradient(normalized, *spacing, edge_order=1)
    gradient_magnitude = np.sqrt(sum(component ** 2 for component in gradients))
    gradient_support = finite_mask & (volume > p05)
    gradient_energy = float(
        gradient_magnitude[gradient_support].mean()
        if gradient_support.any() else gradient_magnitude.mean()
    )

    return {
        'uid': uid_from_member(member_name),
        'member': member_name,
        'shape_x': shape[0], 'shape_y': shape[1], 'shape_z': shape[2],
        'spacing_x_mm': spacing[0], 'spacing_y_mm': spacing[1],
        'spacing_z_mm': spacing[2],
        'spacing_anisotropy': max(spacing) / max(min(spacing), 1e-8),
        'fov_x_mm': fov[0], 'fov_y_mm': fov[1], 'fov_z_mm': fov[2],
        'fov_volume_l': float(np.prod(fov) / 1_000_000),
        'voxel_volume_mm3': float(np.prod(spacing)),
        **original_metadata,
        'canonical_orientation': ''.join(nib.aff2axcodes(image.affine)),
        'finite_fraction': float(finite_mask.mean()),
        'zero_fraction': float(np.mean(finite == 0)),
        'positive_fraction': float(np.mean(finite > 0)),
        'intensity_min': float(finite.min()), 'intensity_max': float(finite.max()),
        'p01': float(p01), 'p05': float(p05), 'p50': float(p50),
        'p90': float(p90), 'p95': float(p95), 'p995': float(p995),
        'positive_entropy_bits': safe_entropy(reference),
        'foreground_volume_ml_proxy': float(
            foreground.sum() * np.prod(spacing) / 1000
        ),
        'high_uptake_components_p90': int(component_count),
        'largest_component_fraction_p90': largest_fraction,
        'centroid_x_mm': float(centroid_mm[0]),
        'centroid_y_mm': float(centroid_mm[1]),
        'centroid_z_mm': float(centroid_mm[2]),
        'lr_uptake_ai_proxy': float(asymmetry_abs),
        'lr_uptake_signed_proxy': float(asymmetry_signed),
        'lr_global_ai': float(asymmetry_abs),
        'gradient_energy': gradient_energy,
        'gradient_energy_per_mm': gradient_energy,
        'slice_corr_z': adjacent_slice_correlation(normalized),
    }


records: list[dict[str, object]] = []
failures: list[dict[str, str]] = []
for member in tqdm(members, desc='Auditando NIfTI'):
    try:
        records.append(scan_record(member))
    except Exception as error:
        failures.append({
            'member': member,
            'error': f'{type(error).__name__}: {error}',
        })

manifest = pd.DataFrame(records)
labels = pd.read_csv(LABELS_PATH, dtype={'uid': 'string'})
labels['uid'] = labels['uid'].astype('string')
manifest['uid'] = manifest['uid'].astype('string')
manifest = manifest.merge(
    labels[['uid', 'is_pathologic']],
    on='uid', how='left', validate='one_to_one',
)
display(Markdown(
    f'**Procesados:** `{len(manifest):,}` · **fallos:** `{len(failures):,}`'
))
display(manifest.head().style.hide(axis='index'))

## 3. Familias de adquisición y scores de revisión

La familia se define sin etiqueta por `(shape, spacing)` redondeado. Se calculan por separado
rareza geométrica global y desviación QC dentro de protocolo; así un protocolo válido pero
poco frecuente no se confunde automáticamente con una imagen defectuosa.

In [ ]:
def robust_z(frame: pd.DataFrame) -> pd.DataFrame:
    numeric = frame.replace([np.inf, -np.inf], np.nan).astype(float)
    numeric = numeric.fillna(numeric.median())
    median = numeric.median()
    mad = (numeric - median).abs().median().replace(0, np.nan)
    return (
        (numeric - median) / (1.4826 * mad)
    ).replace([np.inf, -np.inf], np.nan).fillna(0)


manifest['acquisition_signature'] = manifest.apply(
    lambda row: (
        f"{int(row.shape_x)}x{int(row.shape_y)}x{int(row.shape_z)}|"
        f"{row.spacing_x_mm:.3f},{row.spacing_y_mm:.3f},{row.spacing_z_mm:.3f}"
    ),
    axis=1,
)
signature_order = sorted(manifest['acquisition_signature'].unique())
family_by_signature = {
    signature: f'AF{index + 1:03d}'
    for index, signature in enumerate(signature_order)
}
manifest['acquisition_family'] = manifest['acquisition_signature'].map(
    family_by_signature
)
family_sizes = manifest['acquisition_family'].value_counts()
manifest['acquisition_family_size'] = manifest['acquisition_family'].map(
    family_sizes
).astype(int)
manifest['is_rare_acquisition_family'] = (
    manifest['acquisition_family_size'] < MIN_PROTOCOL_FAMILY_SIZE
)

geometry_frame = pd.DataFrame({
    'log_voxel_volume': np.log1p(manifest['voxel_volume_mm3']),
    'log_fov_volume': np.log1p(manifest['fov_volume_l']),
    'spacing_anisotropy': manifest['spacing_anisotropy'],
    'grid_aspect_xy': manifest['shape_x'] / manifest['shape_y'].clip(lower=1),
    'grid_aspect_zx': manifest['shape_z'] / manifest['shape_x'].clip(lower=1),
})
geometry_z = robust_z(geometry_frame)
manifest['global_geometry_outlier_score'] = np.sqrt(
    (geometry_z ** 2).mean(axis=1)
)

qc_frame = pd.DataFrame({
    'zero_fraction': manifest['zero_fraction'],
    'entropy': manifest['positive_entropy_bits'],
    'log_components': np.log1p(manifest['high_uptake_components_p90']),
    'largest_component_fraction': manifest['largest_component_fraction_p90'],
    'log_gradient_per_mm': np.log1p(manifest['gradient_energy_per_mm']),
    'slice_incoherence': 1 - manifest['slice_corr_z'],
})
global_qc_z = robust_z(qc_frame)
within_qc_z = global_qc_z.copy()
for family, indices in manifest.groupby('acquisition_family').groups.items():
    if len(indices) >= MIN_PROTOCOL_FAMILY_SIZE:
        within_qc_z.loc[indices] = robust_z(qc_frame.loc[indices])

manifest['within_protocol_qc_score'] = np.sqrt((within_qc_z ** 2).mean(axis=1))
manifest['technical_outlier_score'] = np.sqrt(
    (
        manifest['global_geometry_outlier_score'] ** 2
        + manifest['within_protocol_qc_score'] ** 2
    ) / 2
)

protocol_summary = (
    manifest.groupby(['acquisition_family', 'acquisition_signature'], as_index=False)
    .agg(
        n=('uid', 'size'),
        n_labeled=('is_pathologic', 'count'),
        pathologic_rate=('is_pathologic', 'mean'),
        median_qc_score=('within_protocol_qc_score', 'median'),
        median_gradient_per_mm=('gradient_energy_per_mm', 'median'),
        median_slice_corr=('slice_corr_z', 'median'),
    )
    .sort_values('n', ascending=False)
)

review_columns = [
    'uid', 'is_pathologic', 'acquisition_family', 'acquisition_family_size',
    'global_geometry_outlier_score', 'within_protocol_qc_score',
    'technical_outlier_score', 'zero_fraction', 'gradient_energy_per_mm',
    'slice_corr_z',
]
display(Markdown('### Familias de adquisición'))
display(protocol_summary.head(25).style.hide(axis='index'))
display(Markdown('### Casos priorizados para revisión técnica'))
display(
    manifest.nlargest(20, 'technical_outlier_score')[review_columns]
    .style.format({
        'global_geometry_outlier_score': '{:.2f}',
        'within_protocol_qc_score': '{:.2f}',
        'technical_outlier_score': '{:.2f}',
    })
    .hide(axis='index')
)

In [ ]:
integrity = pd.DataFrame({
    'indicador': [
        'Exámenes auditados', 'UID duplicados', 'Sin etiqueta',
        'Fracción finita < 1', 'Spacing no positivo',
        'Affine casi singular', 'Familias de adquisición', 'Familias raras',
    ],
    'valor': [
        len(manifest), int(manifest['uid'].duplicated().sum()),
        int(manifest['is_pathologic'].isna().sum()),
        int((manifest['finite_fraction'] < 1).sum()),
        int((manifest[['spacing_x_mm', 'spacing_y_mm', 'spacing_z_mm']] <= 0)
            .any(axis=1).sum()),
        int((manifest['original_affine_determinant'].abs() < 1e-8).sum()),
        int(manifest['acquisition_family'].nunique()),
        int(manifest.loc[manifest['is_rare_acquisition_family'],
                         'acquisition_family'].nunique()),
    ],
})
display(integrity.style.hide(axis='index'))

figure, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(
    data=manifest, x='spacing_x_mm', hue='is_pathologic',
    element='step', ax=axes[0, 0],
)
axes[0, 0].set_title('Spacing X por clase')
sns.scatterplot(
    data=manifest, x='fov_x_mm', y='fov_z_mm',
    hue='acquisition_family', legend=False, ax=axes[0, 1],
)
axes[0, 1].set_title('Cobertura física por familia de adquisición')
sns.boxplot(
    data=manifest, x='is_pathologic', y='lr_uptake_ai_proxy', ax=axes[1, 0]
)
axes[1, 0].set_title('Asimetría de captación global (proxy)')
sns.scatterplot(
    data=manifest, x='gradient_energy_per_mm', y='slice_corr_z',
    hue='is_pathologic', ax=axes[1, 1],
)
axes[1, 1].set_title('Gradiente físico y coherencia entre cortes')
plt.tight_layout()
plt.show()

## 4. Revisión visual triplanar y MIP

Las MIP toman el máximo por eje; no son promedios. El visor abre un NIfTI a la vez y no
persiste imágenes.

In [ ]:
def plot_scan_qc(uid: str) -> None:
    member = member_by_uid[str(uid)]
    volume, image, _ = load_canonical_volume(member)
    positive = volume[np.isfinite(volume) & (volume > 0)]
    values = positive if positive.size else volume[np.isfinite(volume)]
    vmin, vmax = np.percentile(values, [1, 99.5])
    display_volume = np.clip(volume, vmin, vmax)
    weights = np.where(volume > np.percentile(values, 50), display_volume - vmin, 0)
    center = np.asarray(ndimage.center_of_mass(weights), dtype=float)
    if not np.all(np.isfinite(center)):
        center = (np.asarray(volume.shape) - 1) / 2
    x, y, z = np.clip(
        np.rint(center).astype(int), 0, np.asarray(volume.shape) - 1
    )
    views = [
        ('Sagital', display_volume[x, :, :]),
        ('Coronal', display_volume[:, y, :]),
        ('Axial', display_volume[:, :, z]),
        ('MIP X', display_volume.max(axis=0)),
        ('MIP Y', display_volume.max(axis=1)),
        ('MIP Z', display_volume.max(axis=2)),
    ]
    figure, axes = plt.subplots(2, 3, figsize=(14, 9))
    for axis, (title, plane) in zip(axes.ravel(), views):
        axis.imshow(np.rot90(plane), cmap='hot', vmin=vmin, vmax=vmax)
        axis.set_title(title)
        axis.axis('off')
    spacing = tuple(round(float(v), 3) for v in image.header.get_zooms()[:3])
    figure.suptitle(
        f'{uid} · shape={volume.shape} · spacing={spacing} mm', fontsize=14
    )
    plt.tight_layout()
    plt.show()


default_uid = str(manifest.nlargest(1, 'technical_outlier_score').iloc[0]['uid'])
uid_selector = widgets.Dropdown(
    options=sorted(member_by_uid), value=default_uid, description='UID:',
    layout=widgets.Layout(width='420px'),
    style={'description_width': '60px'},
)
viewer_output = widgets.interactive_output(plot_scan_qc, {'uid': uid_selector})
display(widgets.VBox([uid_selector, viewer_output]))

## 5. Persistencia y linaje

`image_qc_manifest.csv` conserva todos los casos, incluidos los raros o dudosos. Ningún
score elimina exámenes. `acquisition_family_summary.csv` alimenta la validación agrupada del
notebook 05.

In [ ]:
if SAVE_PRIVATE_OUTPUTS:
    PRIVATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(PRIVATE_OUTPUT_DIR / 'image_qc_manifest.csv', index=False)
    pd.DataFrame(failures).to_csv(
        PRIVATE_OUTPUT_DIR / 'image_qc_failures.csv', index=False
    )
    protocol_summary.to_csv(
        PRIVATE_OUTPUT_DIR / 'acquisition_family_summary.csv', index=False
    )
    (PRIVATE_OUTPUT_DIR / 'image_qc_config.json').write_text(
        json.dumps({
            'archive_name': NIFTI_ARCHIVE.name,
            'max_scans': MAX_SCANS,
            'random_seed': RANDOM_SEED,
            'canonical_orientation': 'RAS',
            'gradient_contract': (
                'mean physical gradient magnitude per mm inside voxels above p05'
            ),
            'acquisition_family_contract': 'exact shape plus spacing rounded to 0.001 mm',
            'no_automatic_exclusions': True,
        }, indent=2),
        encoding='utf-8',
    )
    display(Markdown(f'Guardado localmente en `{PRIVATE_OUTPUT_DIR}`.'))
else:
    display(Markdown('Persistencia desactivada.'))